# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [4]:
import sys
print(sys.executable)

/Users/momokasakamoto/ai/projects/tinyml-arduino/bin/python


In [5]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [6]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [8]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [9]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = wine.data
y = wine.target

In [10]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)


In [11]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [16]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y_train, num_classes=num_classes)
y_test = to_categorical(y_test, num_classes=num_classes)

In [17]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])


In [18]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/20
13/13 [==============================] - 0s 11ms/step - loss: 1.0344 - accuracy: 0.4545 - val_loss: 0.8287 - val_accuracy: 0.7600
Epoch 2/20
13/13 [==============================] - 0s 2ms/step - loss: 0.7157 - accuracy: 0.8182 - val_loss: 0.6084 - val_accuracy: 0.8400
Epoch 3/20
13/13 [==============================] - 0s 2ms/step - loss: 0.5103 - accuracy: 0.9495 - val_loss: 0.4517 - val_accuracy: 0.8800
Epoch 4/20
13/13 [==============================] - 0s 2ms/step - loss: 0.3610 - accuracy: 0.9798 - val_loss: 0.3432 - val_accuracy: 0.9200
Epoch 5/20
13/13 [==============================] - 0s 2ms/step - loss: 0.2622 - accuracy: 0.9798 - val_loss: 0.2686 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 2ms/step - loss: 0.1919 - accuracy: 0.9798 - val_loss: 0.2104 - val_accuracy: 1.0000
Epoch 7/20
13/13 [==============================] - 0s 9ms/step - loss: 0.1454 - accuracy: 0.9899 - val_loss: 0.1646 - val_accuracy: 1.0000
Epoch 8/20
13/13 [=

In [19]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

from sklearn.metrics import classification_report, confusion_matrix
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print("Test Accuracy:", test_accuracy)
print("\nClassification Report:")
print(classification_report(y_true, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 1ms/step
Test Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [21]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes
import os
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_model)
model_size_kb = os.path.getsize("model_base.tflite") / 1024
print("TFLite model size:", model_size_kb, "KB")

INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpi6qu3x_b/assets


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpi6qu3x_b/assets


TFLite model size: 14.1015625 KB


2026-05-20 20:17:24.583178: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 20:17:24.583197: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 20:17:24.583341: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpi6qu3x_b
2026-05-20 20:17:24.584157: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 20:17:24.584175: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpi6qu3x_b
2026-05-20 20:17:24.586316: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 20:17:24.637334: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpi6qu3x_b
2026-05-

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [29]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        
    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_model = converter.convert()
    
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    y_pred = []
    y_true = np.argmax(y_test_cat, axis=1)

    for i in range(len(X_test)):
        input_data = X_test[i:i+1].astype(np.float32)

        if input_details[0]["dtype"] in [np.int8, np.uint8]:
            input_scale, input_zero_point = input_details[0]["quantization"]
            input_data = input_data / input_scale + input_zero_point
            input_data = np.round(input_data).astype(input_details[0]["dtype"])

        interpreter.set_tensor(input_details[0]["index"], input_data)
        interpreter.invoke()

        output_data = interpreter.get_tensor(output_details[0]["index"])

        if output_details[0]["dtype"] in [np.int8, np.uint8]:
            output_scale, output_zero_point = output_details[0]["quantization"]
            output_data = output_scale * (output_data.astype(np.float32) - output_zero_point)

        y_pred.append(np.argmax(output_data))

    # Step 4: Report results.
    def file_size_kb(filename):
        return os.path.getsize(filename) / 1024
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    # <-- Enter your code here: print classification_report and confusion_matrix <--#
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


In [30]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test, y_test, 'int8', 'model_int8.tflite')

quantize_and_evaluate(model, X_test, y_test, 'float16', 'model_float16.tflite')

quantize_and_evaluate(model, X_test, y_test, 'dynamic', 'model_dynamic.tflite')


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpyvehml85/assets


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpyvehml85/assets
/Users/momokasakamoto/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 20:38:05.507275: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 20:38:05.507293: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 20:38:05.507451: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpyvehml85
2026-05-20 20:38:05.508152: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 20:38:05.508162: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jz/8pdsq_


INT8 TFLite model size: 5.76 KB

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp8uvdo4cd/assets


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp8uvdo4cd/assets
2026-05-20 20:38:05.921558: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 20:38:05.921574: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 20:38:05.921725: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp8uvdo4cd
2026-05-20 20:38:05.922438: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 20:38:05.922444: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp8uvdo4cd
2026-05-20 20:38:05.924475: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 20:38:05.963063: I tensorflow/cc/saved_model/loader.cc:217] Running initialization


FLOAT16 TFLite model size: 9.00 KB

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp_oivjp5d/assets


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp_oivjp5d/assets



DYNAMIC TFLite model size: 8.20 KB

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        19
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-05-20 20:38:06.370982: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 20:38:06.371004: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 20:38:06.371137: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp_oivjp5d
2026-05-20 20:38:06.371822: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 20:38:06.371828: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp_oivjp5d
2026-05-20 20:38:06.374024: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 20:38:06.411369: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp_oivjp5d
2026-05-

## Problem 1 - Part (c)

### Pruning

In [31]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

import tensorflow_model_optimization as tfmot

# Step 1: Define pruning schedule

batch_size = 8
epochs = 20

end_step = int(np.ceil(len(X_train) / batch_size) * epochs)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

In [32]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude
pruned_model = tf.keras.Sequential([
    prune_low_magnitude(
        tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        tf.keras.layers.Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        tf.keras.layers.Dense(3, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

In [33]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep()
]

history = pruned_model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks
)

Epoch 1/10
13/13 [==============================] - 1s 10ms/step - loss: 0.9973 - accuracy: 0.4646 - val_loss: 0.7967 - val_accuracy: 0.8000
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 0.7065 - accuracy: 0.8889 - val_loss: 0.5723 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 3ms/step - loss: 0.5142 - accuracy: 0.9495 - val_loss: 0.4212 - val_accuracy: 0.9200
Epoch 4/10
13/13 [==============================] - 0s 3ms/step - loss: 0.3819 - accuracy: 0.9596 - val_loss: 0.3183 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 3ms/step - loss: 0.2891 - accuracy: 0.9596 - val_loss: 0.2545 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 3ms/step - loss: 0.2263 - accuracy: 0.9697 - val_loss: 0.2133 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 3ms/step - loss: 0.1773 - accuracy: 0.9697 - val_loss: 0.1804 - val_accuracy: 0.9600
Epoch 8/10
13/13 [=

In [35]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.
def file_size_kb(filename):
    return os.path.getsize(filename) / 1024

stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)
tflite_pruned_model = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned_model)

print("Pruned TFLite model size:", file_size_kb("model_pruned.tflite"), "KB")

INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpb6wg_zhh/assets


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpb6wg_zhh/assets


Pruned TFLite model size: 14.140625 KB


2026-05-20 20:51:35.171205: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 20:51:35.171229: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 20:51:35.171416: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpb6wg_zhh
2026-05-20 20:51:35.171897: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 20:51:35.171903: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpb6wg_zhh
2026-05-20 20:51:35.173554: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 20:51:35.188416: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpb6wg_zhh
2026-05-

In [36]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Step 5: Evaluate the stripped pruned model

y_pred_probs = stripped_pruned_model.predict(X_test)

# Convert probabilities to class labels
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Print classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 2ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.89      0.94        19
           1       0.91      0.95      0.93        21
           2       0.93      1.00      0.97        14

    accuracy                           0.94        54
   macro avg       0.95      0.95      0.95        54
weighted avg       0.95      0.94      0.94        54


Confusion Matrix:
[[17  2  0]
 [ 0 20  1]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [38]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

In [39]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_soft_labels = model.predict(X_train)

4/4 [==============================] - 0s 1ms/step


In [40]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

teacher_preds_soft = teacher_soft_labels

y_train_combined = np.concatenate(
    [y_train, teacher_preds_soft],
    axis=1
)

alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    return alpha * hard_loss + (1 - alpha) * soft_loss

In [41]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history = student_model.fit(
    X_train,
    y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2
)

Epoch 1/10
13/13 [==============================] - 1s 11ms/step - loss: 1.3134 - accuracy: 0.2626 - val_loss: 1.1318 - val_accuracy: 0.5200
Epoch 2/10
13/13 [==============================] - 0s 3ms/step - loss: 1.1199 - accuracy: 0.4646 - val_loss: 0.9614 - val_accuracy: 0.5600
Epoch 3/10
13/13 [==============================] - 0s 3ms/step - loss: 0.9717 - accuracy: 0.5556 - val_loss: 0.8216 - val_accuracy: 0.6000
Epoch 4/10
13/13 [==============================] - 0s 3ms/step - loss: 0.8335 - accuracy: 0.6364 - val_loss: 0.7059 - val_accuracy: 0.8000
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 0.7133 - accuracy: 0.8182 - val_loss: 0.6009 - val_accuracy: 0.8800
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 0.6055 - accuracy: 0.8788 - val_loss: 0.5066 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 3ms/step - loss: 0.5094 - accuracy: 0.9293 - val_loss: 0.4267 - val_accuracy: 0.9600
Epoch 8/10
13/13 [=

In [42]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)

tflite_kd_model = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd_model)

# Print model size in KB
size_kb = os.path.getsize("model_kd.tflite") / 1024

print("Knowledge Distillation TFLite model size:", size_kb, "KB")

INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpslt2tag7/assets


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpslt2tag7/assets


Knowledge Distillation TFLite model size: 6.140625 KB


2026-05-20 20:55:32.330424: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 20:55:32.330450: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 20:55:32.330600: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpslt2tag7
2026-05-20 20:55:32.331540: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 20:55:32.331554: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpslt2tag7
2026-05-20 20:55:32.333869: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 20:55:32.375566: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpslt2tag7
2026-05-

In [43]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_probs = student_model.predict(X_test)

# Convert probabilities to class labels
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Print classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# Print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

2/2 [==============================] - 0s 2ms/step

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97        19
           1       0.95      0.90      0.93        21
           2       0.88      1.00      0.93        14

    accuracy                           0.94        54
   macro avg       0.94      0.95      0.94        54
weighted avg       0.95      0.94      0.94        54


Confusion Matrix:
[[18  1  0]
 [ 0 19  2]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [44]:
# <-- (if needed) Enter your code here <--#

# Further reduction method:
# Pruned model + INT8 quantization

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_pruned_model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

pruned_int8_tflite = converter.convert()

with open("model_pruned_int8.tflite", "wb") as f:
    f.write(pruned_int8_tflite)

print("Pruned + INT8 TFLite model size:", file_size_kb("model_pruned_int8.tflite"), "KB")

INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp1rs4vxwl/assets


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp1rs4vxwl/assets


Pruned + INT8 TFLite model size: 5.8046875 KB


/Users/momokasakamoto/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 21:00:26.012892: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:00:26.012917: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:00:26.013146: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp1rs4vxwl
2026-05-20 21:00:26.013661: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:00:26.013668: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmp1rs4vxwl
2026-05-20 21:00:26.015116: I tensorflow/cc/saved_model/loader.c

In [45]:
quantize_and_evaluate(
    stripped_pruned_model,
    X_test,
    y_test,
    'int8',
    'model_pruned_int8.tflite'
)

INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpe_vrmw4p/assets


INFO:tensorflow:Assets written to: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpe_vrmw4p/assets



INT8 TFLite model size: 5.80 KB

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.89      0.94        19
           1       0.91      0.95      0.93        21
           2       0.93      1.00      0.97        14

    accuracy                           0.94        54
   macro avg       0.95      0.95      0.95        54
weighted avg       0.95      0.94      0.94        54


Confusion Matrix:
[[17  2  0]
 [ 0 20  1]
 [ 0  0 14]]


/Users/momokasakamoto/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 21:01:15.060612: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 21:01:15.060636: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 21:01:15.060807: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpe_vrmw4p
2026-05-20 21:01:15.061532: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 21:01:15.061541: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/jz/8pdsq_393xl0qrj3n_tvkd3w0000gn/T/tmpe_vrmw4p
2026-05-20 21:01:15.063187: I tensorflow/cc/saved_model/loader.c

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
